# Netflix Prize Dataset — Exploratory Data Analysis

This notebook performs a comprehensive EDA of the Netflix Prize dataset, covering:
- Dataset overview and basic statistics
- Rating distributions
- User activity patterns
- Item popularity and long-tail analysis
- Matrix sparsity visualisation
- Temporal trends
- Degree distributions

**Prerequisites**: Run `train_pipeline.py` once first (or at least the data-loading step) so that `data/processed/train.parquet` and `data/processed/test.parquet` exist.

In [ ]:
import sys
sys.path.insert(0, '..')   # allow imports from project root

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path

from src.data_processing import DataProcessor
from src.eda import NetflixEDA

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.2)
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded ✓')

## 1  Load the Dataset

In [ ]:
dp = DataProcessor()

# Load up to 5 M rows (change max_rows=None to load all 100 M)
df = dp.load_data(max_rows=5_000_000, use_cache=True)
movies = dp.load_movie_titles()

print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.dtypes

## 2  Basic Statistics

In [ ]:
eda = NetflixEDA(figures_dir=Path('../reports/figures'))
stats = eda.basic_statistics(df)

stat_df = pd.DataFrame.from_dict(stats, orient='index', columns=['Value'])
stat_df

## 3  Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
counts = df['rating'].value_counts().sort_index()
pct = counts / counts.sum() * 100

palette = sns.color_palette('muted')
axes[0].bar(counts.index, counts.values, color=palette[0], edgecolor='white', width=0.7)
axes[0].set_xlabel('Star Rating'); axes[0].set_ylabel('Count')
axes[0].set_title('Rating Frequency')
axes[0].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

axes[1].bar(pct.index, pct.values, color=palette[1], edgecolor='white', width=0.7)
axes[1].set_xlabel('Star Rating'); axes[1].set_ylabel('%')
axes[1].set_title('Rating Distribution (%)')
for i, v in zip(pct.index, pct.values):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=9)

fig.suptitle('Netflix Prize — Rating Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean rating: {df["rating"].mean():.3f}')
print(f'Median rating: {df["rating"].median():.1f}')
print(f'Std deviation: {df["rating"].std():.3f}')
print('\nObservation: Ratings are left-skewed — users tend to rate movies they like.\n'
      'Rating 4 is the most common, suggesting a positive-bias in user behaviour.')

## 4  Ratings Over Time

In [ ]:
monthly = df.set_index('date').resample('ME')['rating'].count().reset_index()
monthly.columns = ['month', 'count']

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(monthly['month'], monthly['count'], alpha=0.25, color=palette[2])
ax.plot(monthly['month'], monthly['count'], color=palette[2], linewidth=2)
ax.set_xlabel('Month'); ax.set_ylabel('Ratings per Month')
ax.set_title('Monthly Rating Volume (Netflix Prize Dataset)')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
plt.tight_layout()
plt.savefig('../reports/figures/ratings_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

print('Observation: Rating activity increases sharply in 2005, likely reflecting Netflix growth.')

## 5  User Activity Patterns

In [ ]:
user_counts = df.groupby('user_id')['rating'].count()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(user_counts, bins=100, color=palette[3], edgecolor='none', log=True)
axes[0].axvline(user_counts.median(), color='red', linestyle='--', label=f'Median = {user_counts.median():.0f}')
axes[0].axvline(user_counts.mean(), color='orange', linestyle='--', label=f'Mean = {user_counts.mean():.0f}')
axes[0].set_xlabel('Ratings per User'); axes[0].set_ylabel('Users (log scale)')
axes[0].set_title('User Activity Histogram (log-y)')
axes[0].legend()

sorted_c = np.sort(user_counts)
cdf = np.arange(1, len(sorted_c)+1) / len(sorted_c)
axes[1].plot(sorted_c, cdf, color=palette[3], linewidth=2)
axes[1].set_xscale('log')
axes[1].set_xlabel('Ratings per User (log)'); axes[1].set_ylabel('Cumulative Fraction')
axes[1].set_title('CDF of User Activity')
for q in [0.5, 0.9, 0.99]:
    val = np.quantile(user_counts, q)
    axes[1].axhline(q, color='gray', linestyle=':', alpha=0.7)
    axes[1].text(sorted_c.max()*0.5, q+0.01, f'{q:.0%} @ {val:.0f}', fontsize=9)

fig.suptitle('User Activity Patterns', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/user_activity.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Median ratings/user : {user_counts.median():.0f}')
print(f'Mean ratings/user   : {user_counts.mean():.1f}')
print(f'Max ratings/user    : {user_counts.max()}')
print(f'Top 10% users account for {(user_counts[user_counts >= np.quantile(user_counts, 0.9)].sum() / user_counts.sum()):.1%} of all ratings')

## 6  Item Popularity & Long-Tail

In [ ]:
item_counts = df.groupby('item_id')['rating'].count().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.arange(1, len(item_counts)+1)
axes[0].loglog(x, item_counts.values, color=palette[4], linewidth=1.5)
axes[0].fill_between(x, item_counts.values, alpha=0.12, color=palette[4])
axes[0].set_xlabel('Movie Rank (log)'); axes[0].set_ylabel('Ratings (log)')
axes[0].set_title('Long-Tail Item Popularity Curve')
# Annotate top-1% and top-10%
cut1 = int(len(item_counts)*0.01)
cut10 = int(len(item_counts)*0.1)
axes[0].axvline(cut1, color='red', linestyle='--', alpha=0.7, label=f'Top 1% ({cut1} movies)')
axes[0].axvline(cut10, color='orange', linestyle='--', alpha=0.7, label=f'Top 10% ({cut10} movies)')
axes[0].legend(fontsize=9)

top20 = item_counts.head(20).reset_index()
top20 = top20.merge(movies[['item_id','title']], on='item_id', how='left')
top20['label'] = top20['title'].fillna(top20['item_id'].astype(str)).str[:30]
axes[1].barh(range(20), top20['rating'].values[::-1], color=palette[0])
axes[1].set_yticks(range(20))
axes[1].set_yticklabels(top20['label'].values[::-1], fontsize=8)
axes[1].set_xlabel('Number of Ratings')
axes[1].set_title('Top-20 Most-Rated Movies')

fig.suptitle('Item Popularity Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/item_popularity.png', dpi=150, bbox_inches='tight')
plt.show()

top1pct_share = item_counts.head(int(len(item_counts)*0.01)).sum() / item_counts.sum()
print(f'Top 1% of movies account for {top1pct_share:.1%} of all ratings')
print('Observation: Classic power-law / long-tail — a small number of blockbusters dominate,\n'
      'while thousands of niche movies have very few ratings. This sparsity is the core challenge.')

## 7  Matrix Sparsity Heatmap

In [ ]:
rng = np.random.default_rng(42)
sample_users = rng.choice(df['user_id'].unique(), size=200, replace=False)
sample_items = rng.choice(df['item_id'].unique(), size=200, replace=False)

sub = df[df['user_id'].isin(sample_users) & df['item_id'].isin(sample_items)]
matrix = sub.pivot_table(index='user_id', columns='item_id', values='rating', aggfunc='mean')

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    matrix.fillna(0),
    cmap='YlOrRd', ax=ax,
    xticklabels=False, yticklabels=False,
    linewidths=0,
    cbar_kws={'label': 'Rating (0 = missing)'}
)
sparsity = matrix.isna().values.mean()
ax.set_title(f'User–Item Matrix Sample (200×200)\nSparsity = {sparsity:.1%}  |  White = missing',
             fontsize=12)
ax.set_xlabel('Movies (sampled)'); ax.set_ylabel('Users (sampled)')
plt.tight_layout()
plt.savefig('../reports/figures/sparsity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

n_users = df['user_id'].nunique(); n_items = df['item_id'].nunique(); n = len(df)
global_sparsity = 1 - n / (n_users * n_items)
print(f'Global matrix sparsity: {global_sparsity:.4%}')
print('Observation: The matrix is extremely sparse — the core challenge that\n'
      'makes collaborative filtering non-trivial.')

## 8  Degree Distributions

In [ ]:
user_deg = df.groupby('user_id')['item_id'].count()
item_deg = df.groupby('item_id')['user_id'].count()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, deg, label, color in [
    (axes[0], user_deg, 'User (# movies rated)', palette[2]),
    (axes[1], item_deg, 'Movie (# users rated it)', palette[3])
]:
    ax.hist(np.log1p(deg), bins=60, color=color, edgecolor='none')
    ax.axvline(np.log1p(deg.median()), color='red', linestyle='--',
               label=f'Median = {deg.median():.0f}')
    ax.set_xlabel(f'log(1 + {label})')
    ax.set_ylabel('Count')
    ax.set_title(f'{label} Degree Distribution')
    ax.legend()

fig.suptitle('Degree Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/degree_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 9  Average Rating by Year of Movie Release

In [ ]:
df_yr = df.merge(movies[['item_id','year']], on='item_id', how='left')
df_yr = df_yr.dropna(subset=['year'])
df_yr['year'] = pd.to_numeric(df_yr['year'], errors='coerce')
df_yr = df_yr.dropna(subset=['year'])

yr_stats = df_yr.groupby('year').agg(count=('rating','count'), mean=('rating','mean')).reset_index()
yr_stats = yr_stats[yr_stats['count'] >= 500]  # filter low-count years

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax1.bar(yr_stats['year'], yr_stats['count'], alpha=0.35, color=palette[1], label='# Ratings')
ax2.plot(yr_stats['year'], yr_stats['mean'], color=palette[0], marker='o', ms=4, linewidth=2, label='Avg Rating')
ax1.set_xlabel('Year of Movie Release')
ax1.set_ylabel('Number of Ratings'); ax2.set_ylabel('Average Rating')
ax2.set_ylim(2.5, 5)
ax1.set_title('Rating Volume & Average Rating by Movie Release Year')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')
plt.tight_layout()
plt.savefig('../reports/figures/rating_by_release_year.png', dpi=150, bbox_inches='tight')
plt.show()

print('Observation: Classic movies (older years) tend to have higher average ratings\n'
      'despite fewer ratings — survivor bias (only the best old movies are still rated).')

## 10  Key EDA Takeaways

| Finding | Implication |
|---|---|
| Global matrix sparsity ≈ 98–99% | Standard CF breaks; latent factor models handle sparsity better |
| Ratings are left-skewed (mode = 4) | Models must handle positive bias; normalisation / mean-centering is important |
| Long-tail item popularity (power law) | Popular items dominate; cold-start for niche items is a real problem |
| Heavy-tailed user activity | Top 10% of users produce ~50% of ratings; weighting matters |
| Rating volume spikes 2004–2005 | Netflix growth era; temporal splits must respect chronological order to avoid leakage |
| Old movies rated higher on average | Popularity ≠ quality; a blended model (CF + content) may improve recommendations |
